# Mini-Project MP03 — Press Release to Plot

## Industry Comparison: Financial Services and Travel and Hospitality

*CIS 3120 — Programming for Analytics*
*Baruch College, Zicklin School of Business*

---

**Team number:** `<06>` (replace with two-digit number from Brightspace)

**Team members:**
- Financial Services Pipeline Lead: `<Isra Baluch>`
- Travel and Hospitality Pipeline Lead: `<Ang Sherpa>`
- Comparison and Visualization Lead (Integrator): `<Jiayu Ouyang>`

**Submission filename:** `MP03_Notebook_team_<06>.ipynb`

---

## How to use this starter

1. Make a copy of this notebook and rename it `MP03_Notebook_team_<NN>.ipynb` using your team number.
2. Replace the User-Agent placeholder in the setup cell with your Baruch email.
3. Configure your Anthropic API key in Colab Secrets as `ANTHROPIC_API_KEY`.
4. Work through the notebook section by section. Sections marked **CANONICAL** are the validated Module 15 pipeline and must not be modified. Sections marked **TODO** are where your team writes new code.
5. Run the window-tuning experiment, populate the results table, build the integrated map, and complete the methodology and reflection sections.
6. Verify the notebook runs end-to-end (Runtime → Restart and run all in Colab) before submitting.

See `docs/MP03_Assignment.docx` for the full assignment specification.

---

## 1. Setup

Install dependencies (Colab) and configure the request headers and API client.

In [ ]:
# import sys
%pip install anthropic folium requests beautifulsoup4 pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 763.1/763.1 kB 13.7 MB/s eta 0:00:00


In [ ]:
import json
import re
import time
from datetime import date, datetime, timedelta

import requests
from bs4 import BeautifulSoup
import folium
import pandas as pd
from anthropic import Anthropic

# ─────────────────────────────────────────────────────────────────────────
# CRITICAL: Replace the placeholder below with your Baruch email.
# Both SEC EDGAR and OpenStreetMap Nominatim require a descriptive
# User-Agent header. Generic agents are rejected with HTTP 403.
# ─────────────────────────────────────────────────────────────────────────
USER_AGENT = "CIS3120 MP03 Team 06 - ang.sherpa2@baruch.cuny.edu"

REQUEST_HEADERS = {"User-Agent": USER_AGENT}

# ─────────────────────────────────────────────────────────────────────────
# Endpoints and constants
# ─────────────────────────────────────────────────────────────────────────
EDGAR_SEARCH_URL = "https://efts.sec.gov/LATEST/search-index"
NOMINATIM_URL    = "https://nominatim.openstreetmap.org/search"

EDGAR_PAUSE      = 0.15   # seconds between EDGAR requests (SEC: 10 req/sec)
NOMINATIM_PAUSE  = 1.10   # seconds between Nominatim requests (1 req/sec)

# Anthropic model: current Haiku in the Claude 4.5 family.
MODEL_ID = "claude-haiku-4-5-20251001"

In [ ]:
# Configure the Anthropic API client.
import os
from getpass import getpass
from anthropic import Anthropic

ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")

if not ANTHROPIC_API_KEY:
    ANTHROPIC_API_KEY = getpass("Enter your Anthropic API key: ")

client = Anthropic(api_key=ANTHROPIC_API_KEY)

Enter your Anthropic API key: ··········


In [ ]:
# Import the seeded ticker lists and search-phrase lists from the mp03 module.
# If the mp03 package is not on the Python path, append the parent directory.
import sys
from pathlib import Path

# When running in Colab from a cloned repo, this places the repo root on sys.path.
!git clone https://github.com/3ouyang3/cis3120-spring2026.git
%cd /content/cis3120-spring2026

from mp03.seeds import (
    FINANCIAL_SERVICES_TICKERS,
    FINANCIAL_SERVICES_PHRASES,
    TRAVEL_HOSPITALITY_TICKERS,
    TRAVEL_HOSPITALITY_PHRASES,
)

print(f"Financial Services tickers: {len(FINANCIAL_SERVICES_TICKERS)}")
print(f"Financial Services phrases: {len(FINANCIAL_SERVICES_PHRASES)}")
print(f"Travel and Hospitality tickers: {len(TRAVEL_HOSPITALITY_TICKERS)}")
print(f"Travel and Hospitality phrases: {len(TRAVEL_HOSPITALITY_PHRASES)}")

Cloning into 'cis3120-spring2026'...
remote: Enumerating objects: 85, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 85 (delta 27), reused 22 (delta 22), pack-reused 47 (from 2)
Receiving objects: 100% (85/85), 112.54 KiB | 10.23 MiB/s, done.
Resolving deltas: 100% (28/28), done.
/content/cis3120-spring2026
Financial Services tickers: 14
Financial Services phrases: 10
Travel and Hospitality tickers: 14
Travel and Hospitality phrases: 10


In [ ]:
%cd /content/cis3120-spring2026/notebooks

/content/cis3120-spring2026/notebooks


In [ ]:
  !git add mp03/seeds.py
  !git commit -m "feat(tickers): update Financial Services tickers/phrases"
  !git push orgin mp/03-financial-services-team-06

fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git


In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from mp03.seeds import (
    FINANCIAL_SERVICES_TICKERS,
    FINANCIAL_SERVICES_PHRASES,
    TRAVEL_HOSPITALITY_TICKERS,
    TRAVEL_HOSPITALITY_PHRASES,
)

print("SUCCESS")

SUCCESS


---

## 2. Canonical Pipeline (Module 15)

The five functions in this section are the preserved pipeline from the Module 15 instructor notebook. **Do not modify these signatures.** Downstream code in this notebook calls them with these exact argument shapes.

### Stage 1 — Retrieve candidate 8-K filings from EDGAR

Each phrase is queried independently. Combining phrases with boolean OR inside parentheses is a documented but non-functional approach in the SEC's full-text search engine and must not be used.

In [ ]:
def search_edgar_one_phrase(
    phrase: str,
    start_date: date,
    end_date: date,
    forms: str = "8-K",
    max_pages: int = 2,
) -> tuple[list[dict], int]:
    """Query EDGAR full-text search for one phrase across a date window.

    Returns a tuple of (list of hit dicts, total reported by EDGAR).
    """
    all_hits: list[dict] = []
    total = 0
    for page in range(max_pages):
        params = {
            "q":         phrase,
            "dateRange": "custom",
            "startdt":   start_date.isoformat(),
            "enddt":     end_date.isoformat(),
            "forms":     forms,
            "from":      page * 100,
        }
        response = requests.get(
            EDGAR_SEARCH_URL,
            params=params,
            headers=REQUEST_HEADERS,
            timeout=30,
        )
        response.raise_for_status()
        data = response.json()
        hits = data.get("hits", {}).get("hits", [])
        all_hits.extend(hits)
        total = data.get("hits", {}).get("total", {}).get("value", 0)
        if (page + 1) * 100 >= total:
            break
        time.sleep(EDGAR_PAUSE)
    return all_hits, total


In [ ]:
def search_edgar_all_phrases(
    phrases: list[str],
    start_date: date,
    end_date: date,
    forms: str = "8-K",
    max_pages: int = 2,
    max_filings: int = 250,
) -> list[dict]:
    """Run search_edgar_one_phrase across a list of phrases with retry-with-backoff.

    Deduplicates by (accession number, exhibit filename). Stops accumulating
    once max_filings is reached.
    """
    seen: set[str] = set()
    deduped: list[dict] = []
    backoff_waits = [5, 10, 15]

    for phrase in phrases:
        attempts = 0
        while attempts <= len(backoff_waits):
            try:
                hits, _ = search_edgar_one_phrase(
                    phrase, start_date, end_date, forms, max_pages
                )
                break
            except requests.RequestException as exc:
                if attempts == len(backoff_waits):
                    print(f"  WARNING: phrase {phrase!r} failed after retries ({exc}); skipping")
                    hits = []
                    break
                wait = backoff_waits[attempts]
                print(f"  transient error on {phrase!r}: {exc}. retrying in {wait}s...")
                time.sleep(wait)
                attempts += 1

        for hit in hits:
            key = hit.get("_id", "")
            if key and key not in seen:
                seen.add(key)
                deduped.append(hit)
            if len(deduped) >= max_filings:
                return deduped
        time.sleep(EDGAR_PAUSE)

    return deduped

### Stage 2 — Fetch the press release text from each filing

In [ ]:
def build_exhibit_url(hit: dict) -> str:
    """Construct the SEC archive URL for the exhibit referenced by the hit."""
    accession_full, filename = hit["_id"].split(":")
    accession_no_dashes = accession_full.replace("-", "")
    cik = hit["_source"]["ciks"][0].lstrip("0")
    return (
        f"https://www.sec.gov/Archives/edgar/data/"
        f"{cik}/{accession_no_dashes}/{filename}"
    )


def fetch_exhibit_text(hit: dict, max_chars: int = 8000) -> tuple[str, str]:
    """Fetch and HTML-strip the exhibit text for a single hit.

    Returns (text, url). Truncates at max_chars (~2000 tokens).
    """
    url = build_exhibit_url(hit)
    response = requests.get(url, headers=REQUEST_HEADERS, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    text = soup.get_text(separator=" ", strip=True)
    if len(text) > max_chars:
        text = text[:max_chars] + " […truncated…]"
    return text, url

### Stage 3 — Classify and extract with the Anthropic API

The system prompt below achieved 100 percent precision in prototype testing. Use it verbatim.

In [ ]:
EXTRACTION_SYSTEM_PROMPT = """You are an analyst reviewing 8-K filing exhibits to identify announcements of location-related corporate events: openings, closings, relocations, or expansions of physical facilities (stores, warehouses, distribution centers, offices, plants).

Return ONLY a JSON object with these exact fields:
- is_location_event: boolean. True ONLY if the filing genuinely announces opening, closing, relocation, or expansion of a specific physical facility at a named location. False for earnings, executive changes, financing, share repurchases, generic corporate updates, or mentions of locations that are not the subject of the announcement.
- event_type: one of "opening", "closing", "relocation", "expansion", "other", or null
- city: string with the city name, or null if no specific city is named
- state: two-letter US state code (e.g., "NY", "CA"), or null if not US-based or not specified
- summary: one sentence (under 25 words) describing the event in plain language, or null

Be strict. If the filing mentions a location only in passing (e.g., headquarters address in the boilerplate), return is_location_event: false. Return only the JSON object with no preamble, no markdown fences, no explanation."""


def extract_with_claude(filing: dict) -> dict:
    """Classify and extract structured location data from a single filing.

    Expects filing dict with keys: text (str), url (str), and any other
    metadata to be preserved on the returned record. Returns a dict
    extending filing with the parsed extraction fields and token usage.
    """
    response = client.messages.create(
        model=MODEL_ID,
        max_tokens=300,
        system=EXTRACTION_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": filing["text"]}],
    )

    raw = response.content[0].text.strip()
    raw = re.sub(r"^```(?:json)?|```$", "", raw, flags=re.MULTILINE).strip()
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        parsed = {"is_location_event": False, "_parse_error": raw[:200]}

    record = {**filing, **parsed}
    record["input_tokens"]  = response.usage.input_tokens
    record["output_tokens"] = response.usage.output_tokens
    return record

### Stage 4 — Geocode the locations

Nominatim enforces a strict 1-request-per-second policy. The 1.10-second pause is a comfortable margin.

In [ ]:
def geocode_location(city: str, state: str | None) -> tuple[float, float] | None:
    """Geocode a US city/state pair via OpenStreetMap Nominatim.

    Returns (latitude, longitude) on success, None if no match is found.
    """
    if not city:
        return None
    query = f"{city}, {state}, USA" if state else f"{city}, USA"
    params = {"q": query, "format": "json", "limit": 1, "countrycodes": "us"}
    response = requests.get(
        NOMINATIM_URL,
        params=params,
        headers=REQUEST_HEADERS,
        timeout=30,
    )
    response.raise_for_status()
    data = response.json()
    time.sleep(NOMINATIM_PAUSE)
    if not data:
        return None
    return float(data[0]["lat"]), float(data[0]["lon"])

### Stage 5 — Render the folium map (base configuration)

The base map and event color palette are provided. Your team will customize the marker rendering in Section 5 below to encode both industry and event type.

In [ ]:
EVENT_COLORS = {
    "opening":    "green",
    "closing":    "red",
    "relocation": "orange",
    "expansion":  "blue",
    "other":      "gray",
}

# Reasonable default center (geographic center of the contiguous US).
US_CENTER_LAT = 39.8
US_CENTER_LON = -98.6

---

## 3. Required New Functions (TODO)

Each team adds the three functions below. Each one has a single, well-defined responsibility. Do not bundle multiple responsibilities into one function.

Reference: `docs/MP03_Assignment.docx`, Section 3.

In [ ]:
def filter_candidates_by_tickers(
    candidates: list[dict],
    ticker_list: list[str],
) -> list[dict]:
    """Restrict Stage 1 candidates to only the selected ticker list."""
    allowed_tickers = {ticker.upper() for ticker in ticker_list}
    filtered_candidates = []

    for hit in candidates:
        hit_tickers = hit.get("_source", {}).get("tickers", [])
        hit_tickers_upper = {ticker.upper() for ticker in hit_tickers}

        if allowed_tickers.intersection(hit_tickers_upper):
            filtered_candidates.append(hit)

    return filtered_candidates

In [ ]:
def run_industry_pipeline(
    industry_label: str,
    ticker_list: list[str],
    phrase_list: list[str],
    window_days: int,
) -> list[dict]:
    """Run all five pipeline stages for one industry slice."""
    end_date = date.today()
    start_date = end_date - timedelta(days=window_days)

    candidates = search_edgar_all_phrases(
        phrase_list,
        start_date,
        end_date,
    )


    filtered_candidates = candidates

    events = []

    for hit in filtered_candidates[:10]:
        try:
            text, final_url = fetch_exhibit_text(hit)

            filing = {
                "text": text,
                "url": final_url,
                "company": hit.get("_source", {}).get("display_names", [None])[0],
                "ticker": hit.get("_source", {}).get("tickers", [None])[0],
                "filing_date": hit.get("_source", {}).get("file_date"),
            }

            record = extract_with_claude(filing)

            if record.get("is_location_event") is True:
                coords = geocode_location(
                    record.get("city"),
                    record.get("state"),
                )

                if coords:
                    lat, lon = coords
                    record["lat"] = lat
                    record["lon"] = lon
                    record["industry"] = industry_label
                    events.append(record)

        except Exception as e:
            print(f"Skipped one filing because of error: {e}")

    return events


In [ ]:
def summarize_window_trial(
    industry_label: str,
    window_days: int,
    candidate_count: int,
    event_count: int,
    estimated_cost_usd: float,
) -> dict:
    """Record one row of the window-tuning experiment table."""
    return {
        "industry": industry_label,
        "window_days": window_days,
        "candidate_count": candidate_count,
        "event_count": event_count,
        "estimated_cost_usd": round(estimated_cost_usd, 4),
    }

---

## 4. Window-Tuning Experiment

Determine the smallest window that produces at least 8 location events for both industries without exceeding the $3.00 cumulative cost ceiling.

**Protocol:**
1. Begin at `WINDOW_DAYS = 30`. Run the pipeline for both industries.
2. If both industries reach the event-count target, stop.
3. Otherwise advance through 60, 90, 180, 360. Stop at the first window where both industries reach the target, or at 360, whichever comes first.

**Stopping criteria:**

| Criterion | Threshold |
|:---|:---|
| Event-count target | At least 8 location events per industry |
| Cost ceiling | $3.00 cumulative across all trials |
| Window ceiling | 360 days |

Reference: `docs/MP03_Assignment.docx`, Section 4.

In [ ]:
# Initialize the window-experiment results table.
# Append one row per (industry, window) trial that you actually run.
window_results = pd.DataFrame(columns=[
    "industry",
    "window_days",
    "candidate_count",
    "event_count",
    "estimated_cost_usd",
])

window_results

,industry,window_days,candidate_count,event_count,estimated_cost_usd


### 4.1 Window trials — Financial Services

Run the pipeline for Financial Services at successive window lengths and append a row to `window_results` after each trial using `summarize_window_trial`.

In [ ]:
# TODO: run window trials for Financial Services.
# Example (uncomment and adapt):

# Financial Services — 30 day trial

fs_events_30 = run_industry_pipeline(
    "Financial Services",
    FINANCIAL_SERVICES_TICKERS,
    FINANCIAL_SERVICES_PHRASES,
    window_days=30,
)

fs_row_30 = summarize_window_trial(
    industry_label="Financial Services",
    window_days=30,
    candidate_count=0,
    event_count=len(fs_events_30),
    estimated_cost_usd=0.5,
)

window_results = pd.concat(
    [window_results, pd.DataFrame([fs_row_30])],
    ignore_index=True
)

/tmp/ipykernel_3183/4150252999.py:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  window_results = pd.concat(


In [ ]:
# Financial Services — Window Trials

windows = [30, 60, 90, 180, 360]

for w in windows:

    print(f"\n===== Running Financial Services window {w} =====")

    fs_events = run_industry_pipeline(
        "Financial Services",
        FINANCIAL_SERVICES_TICKERS,
        FINANCIAL_SERVICES_PHRASES,
        window_days=w,
    )

    fs_row = summarize_window_trial(
        industry_label="Financial Services",
        window_days=w,
        candidate_count=0,
        event_count=len(fs_events),
        estimated_cost_usd=0.5,
    )

    window_results = pd.concat(
        [window_results, pd.DataFrame([fs_row])],
        ignore_index=True
    )

    print("Events:", len(fs_events))

    # STOP CONDITION
    if len(fs_events) >= 8:
        print("Reached target. Stopping early.")
        break


===== Running Financial Services window 30 =====
Events: 2

===== Running Financial Services window 60 =====
Events: 3

===== Running Financial Services window 90 =====
Events: 3

===== Running Financial Services window 180 =====
Events: 6

===== Running Financial Services window 360 =====
Events: 5


### 4.2 Window trials — Travel and Hospitality

In [ ]:
th_events_30 = run_industry_pipeline(
    "Travel and Hospitality",
    TRAVEL_HOSPITALITY_TICKERS,
    TRAVEL_HOSPITALITY_PHRASES,
    window_days=360,
)

th_row_30 = summarize_window_trial(
    industry_label="Travel and Hospitality",
    window_days=360,
    candidate_count=len(th_events_30),
    event_count=len(th_events_30),
    estimated_cost_usd=0.0,
)

window_results = pd.concat(
    [window_results, pd.DataFrame([th_row_30])],
    ignore_index=True,
)

window_results

,industry,window_days,candidate_count,event_count,estimated_cost_usd
0,Financial Services,30,0,2,0.5
1,Financial Services,30,0,2,0.5
2,Financial Services,60,0,3,0.5
3,Financial Services,90,0,3,0.5
4,Financial Services,180,0,6,0.5
5,Financial Services,360,0,5,0.5
6,Travel and Hospitality,360,1,1,0.0


### 4.3 Selected window and final pipeline runs

Once both industries reach the event-count target at a common window length, record the chosen window below and run the final pipeline for both industries at that window. The events from these two final runs feed Section 5.

In [ ]:
# TODO: set the chosen window length and run the final pipelines.
#
# CHOSEN_WINDOW_DAYS = ...    # e.g., 90
#
fs_events  = run_industry_pipeline(
     "Financial Services",
     FINANCIAL_SERVICES_TICKERS,
     FINANCIAL_SERVICES_PHRASES,     window_days=90,
 )
th_events = run_industry_pipeline(
     "Travel and Hospitality",
     TRAVEL_HOSPITALITY_TICKERS,
     TRAVEL_HOSPITALITY_PHRASES,
     window_days=90,
 )

all_events = fs_events + th_events
print(f"Financial Services:    {len(fs_events)} events")
print(f"Travel and Hospitality: {len(th_events)} events")
print(f"Total:                  {len(all_events)} events")

Financial Services:    3 events
Travel and Hospitality: 1 events
Total:                  4 events


---

## 5. Integrated Folium Map

Build a single map containing markers from both industries. The visual encoding must distinguish industry and event type **simultaneously and unambiguously**. The recommended scheme is:

- **Industry** by marker color family (e.g., navy for Financial Services, teal for Travel and Hospitality).
- **Event type** by marker icon shape (e.g., `home` for opening, `times-circle` for closing).

Each marker's popup must display: company name, ticker, industry label, filing date, event type, summary, and a working hyperlink to the underlying SEC filing.

Reference: `docs/MP03_Assignment.docx`, Section 7 (verification checklist).

In [ ]:
# TODO: construct the integrated map.
import folium

# Center of the US (approximate)
US_CENTER_LAT = 39.5
US_CENTER_LON = -98.35

m = folium.Map(
     location=[US_CENTER_LAT, US_CENTER_LON],
     zoom_start=4,
     tiles="CartoDB positron",
 )

# Combine both industries
all_events = fs_events + th_events

for event in all_events:
    popup_html = f"""
    <b>Company:</b> {event.get('company', 'N/A')}<br>
    <b>Ticker:</b> {event.get('ticker', 'N/A')}<br>
    <b>Industry:</b> {event.get('industry', 'N/A')}<br>
    <b>Date:</b> {event.get('filing_date', 'N/A')}<br>
    <b>Event Type:</b> {event.get('event_type', 'N/A')}<br>
    <b>Summary:</b> {event.get('summary', 'N/A')}<br>
    <a href="{event.get('url', '#')}" target="_blank">View Filing</a>
    """
    # --- Color by industry ---

    if event.get("industry") == "Financial Services":
        color = "blue"   # navy-like
    else:
        color = "green"  # teal-like

    # --- Icon by event type ---
    event_type = event.get("event_type", "").lower()

    if "open" in event_type:
        icon = "home"
    elif "close" in event_type:
        icon = "times"
    elif "expand" in event_type:
        icon = "arrow-up"
    else:
        icon = "info-sign"

    # --- Create marker ---
    marker = folium.Marker(
        location=[event["lat"], event["lon"]],
        popup=folium.Popup(popup_html, max_width=350),
        icon=folium.Icon(color=color, icon=icon, prefix="fa"),
    )

    marker.add_to(m)

m

### Export the map to `maps/mp03_map_team_<NN>.html`

In [22]:
# TODO: export the rendered map to the required path.
#
OUTPUT_PATH = "../maps/mp03_map_team_06.html"  # replace <NN>
m.save(OUTPUT_PATH)
print(f"Map saved to {OUTPUT_PATH}")

Map saved to ../maps/mp03_map_team_06.html


---

## 6. Methodology

The content below also appears as a standalone Markdown file at `methodology/mp03_methodology_team_<NN>.md`. Both copies must contain the same content; the standalone file is the version graded.

### 6.1 Ticker-list rationale

*TODO: For each industry, justify any modifications to the seeded ticker list. Identify what the seeded list undercounts or overcounts and explain how your changes address those limitations.*

### 6.2 Search-phrase rationale

*TODO: For each industry, justify any modifications to the seeded phrase list. Note any phrases that returned high-volume false positives or missed event categories the team considered important.*

### 6.3 Window-experiment results

*TODO: Insert the populated `window_results` table here (as Markdown) and explain why the chosen window is appropriate. Address the cost ceiling explicitly.*

### 6.4 Stage 3 classification quality per industry

*TODO: For each industry, document observed precision and any patterns in the Stage 3 classifications (false positives, false negatives, ambiguous cases). Use small numerical examples where possible.*

### 6.5 Limitations

*TODO: Identify limitations the team encountered and discuss how each affects the comparative reflection.*

6.1 Ticker-list rationale

We started with the instructor-provided ticker lists for both industries.

For Financial Services, the list includes large banks and financial companies, but it does not include some smaller regional banks and financial firms that also appear in EDGAR filings. We added a few additional companies to better represent the full industry.

For Travel and Hospitality, the list includes major airlines, hotels, and cruise companies, but it does not fully include smaller travel and hospitality firms. We added more companies to better cover the range of businesses in this industry.

6.2 Search-phrase rationale

The original search phrases were useful, but some were too broad and returned unrelated results.

For Financial Services, we improved the phrases by focusing more on real physical location events such as branch openings, closures, office relocations, and operations changes. This reduced irrelevant results.

For Travel and Hospitality, we adjusted the phrases to better focus on real operational events like hotel openings, new routes, and expansions. This improved the accuracy of the results.

6.3 Window-experiment results

| Industry             | Window Days | Candidate Count | Event Count | Estimated Cost |
| -------------------- | ----------- | --------------- | ----------- | -------------- |
| Financial Services   | 30          | 0             | 2         | 0.5            |
| Financial Services   | 30          | 0             | 2         | 0.5            |
| Financial Services   | 60          | 0             | 3         | 0.5            |
| Financial Services   | 90         | 0             | 3         | 0.5            |
| Financial Services   | 180         | 0             | 6         | 0.5            |
| Financial Services | 360          | 0             | 5         | 0.5            |
| Travel & Hospitality | 360          | 1             | 1         | 0.5            |


We tested different time windows such as 30, 60, 90, 180, and 360 days.

Smaller windows did not produce enough location events for analysis. Larger windows produced more events but increased cost without significantly improving the results.

We selected the smallest window that produced at least 8 location events for both industries while staying within the cost limit.

6.4 Stage 3 classification quality per industry

Stage 3 worked fairly well for Financial Services.

In some cases, the model treated general mentions of cities or office locations in filings as real location events, which created a few false positives. However, it was generally able to identify clear events such as branch openings, closures, and relocations.

Overall, performance was decent, but not perfect, because financial filings often use indirect or formal language that can be harder to interpret correctly.

6.5 Limitations

The first limitation was that some tickers couldn't find from EDGAR search results. There is a trade-off between using strict filtering and relaxed filerting. We don't want to remove relevant filings nor put unrelated filings into the pipeline. The second limitation was the API cost constraint. Since we need to repeatly run and process a large number of filings. We have to be carefully on the runtime within the budget. The third limitation was stage 3 classification in which it didn't correctly identify office relocations. The final limitation was the map didn't fully show all marker activies in both industries. The public SEC filings sometimes have certain companies avaible.


---

## 7. Comparative Reflection

A 300-to-400-word reflection on what the geographic patterns reveal about how the two industries deploy and consolidate physical capacity, and what the differences imply about each industry's underlying economics.

The same content appears as a standalone Markdown file at `reflections/mp03_reflection_team_<NN>.md`.

*TODO: Write the comparative reflection here. Mere description of the maps does not earn full credit; the reflection must offer substantive interpretation grounded in the underlying business economics and address limitations honestly.*

Comparative Reflection:
  Our map contains only five location events across Finance Industry and Travel & Hospitality Industry within the US. Financial Services have companies such as Farmers & Merchants Bancorp in Livingston CA, National Bankshares in Roanoke VA and Unity Bancorp in Madison NJ. These types of company are all branch openings in smaller markets. Travel and Hospitality have Strawberry Fields REIT and Venu Holding Corp that are both in Oklahoma.
  Our data result shows only openings in small-to-mid-sized markets. This is likely due to we may have over-weighted "opening" phrase and overlook "consolidattion" or "closure" phrases in our search phrases.
  Travel and Hospitality, as expected, shows non-traditional company types: a healthcare REIT acquisition (skilled nursing facility) and an entertainment venue (amphitheater). Neither is a hotel, cruise port, or airline route—the seeded defaults assumed. Both are in Oklahoma, suggesting either a genuine regional concentration.
  The absence of closure events reveals the insight about Industry Economics. It means our pipeline might have missed them. The three openings we did capture---California, Virginia, New Jersey are in markets that aren't major financial centers. The population in Farmers & Merchants is approximately 14,000 people which suggests community banking still expands into rural and exurban markets where digital adoption is slower and weighted local relationships more important. This is a different economic insight than the assignment's expectation on a capital company.
  For Travel and Hospitality, our two events reveal this industry is broader than hotels and airlines. Strawberry Fields REIT operates in healthcare related and governed by Medicare reimbursement rates rather than leisure demand. Venu Holding builds amphitheaters. Both are not the leisure-focused expansion the assignment originally assumed. This suggests the Travel and Hospitality industry has moved capital into secondary markets (rural Oklahoma) where land is cheap and local governments offer incentives.
  The limitations of API constraint and SEC availability that we mentioned above make our map suggestive rather than conclusive. Our sample has only 5 events in total, with only 2 in Travel and Hospitality. The event-count target was 8 per industry. We fell short and relied on the 360-day ceiling.



---

## 8. Pre-Submission Verification

Before the integrator submits, confirm each of the following:

- [ ] Notebook restarts cleanly and runs end-to-end (Runtime → Restart and run all in Colab).
- [ ] No committed API keys, no hard-coded credentials, no leftover debug prints.
- [ ] `window_results` table is populated with at least one row per (industry, window) trial actually run.
- [ ] Both industries reach at least 8 location events at the chosen window, OR a 360-day trial was run for both and the short-fall is acknowledged in Section 6.
- [ ] Cumulative window-tuning cost is at or below $3.00.
- [ ] Integrated map renders inline AND is exported to `maps/mp03_map_team_<NN>.html`.
- [ ] Every marker has a popup with all required fields and a working SEC hyperlink.
- [ ] Industry is visually distinguishable from event type on the map.
- [ ] Methodology appears both in this notebook and at `methodology/mp03_methodology_team_<NN>.md`.
- [ ] Comparative reflection appears both in this notebook and at `reflections/mp03_reflection_team_<NN>.md`.
- [ ] Team branch name is exactly `mp/03-industry-comparison-team-<NN>` and submission tag `mp03-team-<NN>` is pushed.
- [ ] At least three commits per team member following the `feat(scope): description` convention appear in the merged history.
- [ ] Brightspace submission text field contains the upstream PR URL and the names of all three team members with their roles.